# Soybean Strategy Backtest Research

Standalone research notebook for soybean. The strategy logic is inside this notebook: generic A/B tests, price Momentum/MR, annual walk-forward, Cargill overlay, and named-period check.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from grain_futures_strategy import (
    backtest_positions_with_costs,
    build_feature_panels,
    load_train_set,
    split_performance,
)
from research_config import DEFAULT_MARGIN_PER_LOT, REGIME_PERIODS, SPLIT_DATE

DATA_DIR = "train_set"
OOS_START = pd.Timestamp(SPLIT_DATE)
TRAIN_END = pd.Timestamp("2016-01-01")
DD_CAPITAL_USD = 10_000.0
TRADE_COST_PER_LOT = 8.75
HOLDING_COST_RATE = 0.05

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

COMMODITY = "SOYABEAN"

## 1. Load Data And Confirm Cargill Inputs

In [2]:
data = load_train_set(DATA_DIR)
feature_panels, futures_pnl_all = build_feature_panels(data)
trading_index = futures_pnl_all.index
futures_pnl = futures_pnl_all[[COMMODITY]]
panel = feature_panels[COMMODITY].reindex(trading_index).fillna(0.0)

display(pd.DataFrame([{
    "commodity": COMMODITY,
    "rows": len(trading_index),
    "start": trading_index.min(),
    "end": trading_index.max(),
    "has_cargill_inputs": {"cgl_inventory_change", "crush_surprise", "crush_utilization"}.issubset(panel.columns),
    "train_rows": int((trading_index < TRAIN_END).sum()),
    "validation_rows": int(((trading_index >= TRAIN_END) & (trading_index < OOS_START)).sum()),
    "oos_rows": int((trading_index >= OOS_START).sum()),
}]))

,commodity,rows,start,end,has_cargill_inputs,train_rows,validation_rows,oos_rows
0,SOYABEAN,2866,2010-01-04,2020-12-31,True,1565,520,781


## 2. Standalone Helpers

In [3]:
def clean_signal(series, target_index):
    return (
        pd.Series(series, index=target_index)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .clip(-5.0, 5.0)
    )


def family_average(families, target_index):
    signals = []
    for members in families.values():
        signals.extend(members)
    return clean_signal(pd.concat(signals, axis=1).mean(axis=1), target_index)


def equal_family(families, target_index):
    family_signals = [pd.concat(members, axis=1).mean(axis=1) for members in families.values()]
    return clean_signal(pd.concat(family_signals, axis=1).mean(axis=1), target_index)


def positions_from_signal(signal, pnl, commodity, target_daily_vol=75.0, max_abs_lot=0.50):
    signal = clean_signal(signal, pnl.index)
    signal = pd.Series(np.tanh(signal / 2.0), index=pnl.index)
    signal = signal.ewm(halflife=3.0, adjust=False, min_periods=1).mean()
    signal[signal.abs() < 0.05] = 0.0
    vol = pnl[commodity].rolling(60, min_periods=20).std().shift(1).replace(0.0, np.nan)
    lots = (signal * float(target_daily_vol) / vol).clip(-float(max_abs_lot), float(max_abs_lot)).fillna(0.0)
    return pd.DataFrame({commodity: lots}, index=pnl.index)


def backtest_signal(signal, pnl, commodity):
    return backtest_positions_with_costs(
        positions_from_signal(signal, pnl, commodity),
        pnl,
        trade_cost_per_lot=TRADE_COST_PER_LOT,
        holding_cost_rate=HOLDING_COST_RATE,
        margin_per_lot=DEFAULT_MARGIN_PER_LOT,
    )[0]


def active_metrics(bt, mask=None):
    sample_bt = bt if mask is None else bt.loc[mask]
    active = sample_bt["held_gross_exposure"] > 1.0e-12
    pnl = sample_bt.loc[active, "net_pnl"].dropna()
    if len(pnl) == 0:
        return {"days": np.nan, "total_pnl": np.nan, "sharpe": np.nan, "max_drawdown": np.nan, "hit_rate": np.nan, "turnover": np.nan}
    cum = pnl.cumsum()
    dd = cum - cum.cummax()
    vol = pnl.std()
    return {
        "days": float(len(pnl)),
        "total_pnl": float(pnl.sum()),
        "sharpe": np.nan if vol == 0.0 else float(pnl.mean() / vol * np.sqrt(252.0)),
        "max_drawdown": float(dd.min()),
        "hit_rate": float((pnl > 0.0).mean()),
        "turnover": float(sample_bt.loc[pnl.index, "turnover"].mean()),
    }


def metric_row(label, bt):
    train = active_metrics(bt, bt.index < TRAIN_END)
    validation = active_metrics(bt, (bt.index >= TRAIN_END) & (bt.index < OOS_START))
    oos = active_metrics(bt, bt.index >= OOS_START)
    full = active_metrics(bt)
    return {
        "strategy": label,
        "mode": "long_short",
        "train_sharpe": train["sharpe"],
        "validation_sharpe": validation["sharpe"],
        "oos_sharpe": oos["sharpe"],
        "oos_pnl": oos["total_pnl"],
        "oos_dd_pct": oos["max_drawdown"] / DD_CAPITAL_USD * 100.0,
        "full_sharpe": full["sharpe"],
        "turnover": full["turnover"],
    }


def zscore_from_train(x_train, x_row):
    mean = x_train.mean()
    std = x_train.std().replace(0.0, np.nan)
    return ((x_train - mean) / std).clip(-5.0, 5.0).fillna(0.0), ((x_row - mean) / std).clip(-5.0, 5.0).fillna(0.0)


def expanding_ols_prediction(x, y, min_train_days=504, refit_every=21):
    preds = pd.Series(np.nan, index=x.index)
    beta, last_fit = None, None
    for i, date in enumerate(x.index):
        train_mask = (x.index < date) & y.notna()
        if train_mask.sum() < min_train_days:
            continue
        if beta is None or last_fit is None or (i - last_fit) >= refit_every:
            x_train, x_row = zscore_from_train(x.loc[train_mask], x.loc[date])
            design = np.column_stack([np.ones(len(x_train)), x_train.values])
            beta, *_ = np.linalg.lstsq(design, y.loc[train_mask].values.astype(float), rcond=None)
            last_fit = i
        else:
            _, x_row = zscore_from_train(x.loc[train_mask], x.loc[date])
        preds.loc[date] = np.r_[1.0, x_row.values.astype(float)].dot(beta)
    return preds


def kalman_prediction(x, y, min_train_days=504, process_noise=1.0e-5):
    columns = list(x.columns)
    beta = np.zeros(len(columns) + 1)
    covariance = np.eye(len(beta)) * 10.0
    mean = pd.Series(0.0, index=columns)
    var = pd.Series(1.0, index=columns)
    target_var = 1.0
    preds = pd.Series(np.nan, index=x.index)
    n = 0
    for date in x.index:
        row = x.loc[date]
        if n > min_train_days:
            z = ((row - mean) / np.sqrt(var.clip(lower=1.0e-8))).clip(-5.0, 5.0)
            preds.loc[date] = np.r_[1.0, z.values.astype(float)].dot(beta)
        y_value = y.loc[date]
        if pd.notnull(y_value):
            n += 1
            old_mean = mean.copy()
            mean = mean + (row - mean) / float(n)
            var = ((n - 2.0) / max(n - 1.0, 1.0)) * var + ((row - old_mean) * (row - mean)) / max(n - 1.0, 1.0)
            target_var = target_var + (float(y_value) ** 2 - target_var) / float(n)
            if n > min_train_days:
                z = ((row - mean) / np.sqrt(var.clip(lower=1.0e-8))).clip(-5.0, 5.0)
                phi = np.r_[1.0, z.values.astype(float)]
                covariance = covariance + np.eye(len(beta)) * float(process_noise)
                innovation_var = float(phi.dot(covariance).dot(phi) + max(target_var, 1.0))
                gain = covariance.dot(phi) / innovation_var
                beta = beta + gain * float(y_value - phi.dot(beta))
                covariance = covariance - np.outer(gain, phi).dot(covariance)
    return preds


def prediction_to_signal(prediction, target_index):
    mean = prediction.rolling(252, min_periods=60).mean().shift(1)
    std = prediction.rolling(252, min_periods=60).std().shift(1).replace(0.0, np.nan)
    return clean_signal((prediction - mean) / std, target_index)


def select_by_ic_signal(families, pnl, target_index, lookback=504):
    target = pnl.shift(-1)
    family_signals = {name: clean_signal(pd.concat(members, axis=1).mean(axis=1), target_index) for name, members in families.items()}
    out = pd.Series(0.0, index=target_index)
    for date in target_index:
        train = target_index < date
        recent = target.loc[train].tail(lookback)
        if recent.notna().sum() < 120:
            continue
        scores = {}
        for name, signal in family_signals.items():
            aligned = pd.concat([signal.loc[recent.index], recent], axis=1).dropna()
            scores[name] = aligned.iloc[:, 0].corr(aligned.iloc[:, 1]) if len(aligned) > 60 else np.nan
        scores = pd.Series(scores).dropna()
        if scores.empty:
            continue
        out.loc[date] = family_signals[scores.abs().idxmax()].loc[date]
    return clean_signal(out, target_index)


def best_family_by_trend_mr(families, panel, target_index):
    trend_strength = panel["mom_60"].abs()
    threshold = trend_strength.expanding(min_periods=252).median().shift(1)
    trend_regime = (trend_strength > threshold).fillna(False)
    price_trend = pd.concat(families["price_trend"], axis=1).mean(axis=1)
    price_mr = pd.concat(families["price_mr"], axis=1).mean(axis=1)
    return clean_signal(price_trend.where(trend_regime, price_mr), target_index)


def named_period_check(bt):
    rows = []
    for item in REGIME_PERIODS:
        mask = (bt.index >= pd.Timestamp(item["start"])) & (bt.index <= pd.Timestamp(item["end"]))
        m = active_metrics(bt, mask)
        rows.append({
            "period": item["period"],
            "total_pnl": m["total_pnl"],
            "sharpe": m["sharpe"],
            "max_dd_pct": m["max_drawdown"] / DD_CAPITAL_USD * 100.0 if pd.notnull(m["max_drawdown"]) else np.nan,
            "hit_rate": m["hit_rate"],
            "days": m["days"],
        })
    return pd.DataFrame(rows)


## 3. Generic Signal A/B Tests

In [4]:
families_A = {
    "price_trend": [panel["mom_20"], panel["mom_60"]],
    "price_mr": [panel["rev_5"]],
    "curve": [panel["curve_spread"], panel["curve_ratio"], panel["curve_change_20"]],
    "cot": [panel["cot_mm_level"], panel["cot_mm_change"], panel["cot_pm_oi_level"], panel["cot_pm_oi_change"]],
}
families_B = {
    **families_A,
    "physical_public": [-panel["public_inventory_change"], -panel["receipts_change"]],
    "physical_cargill": [-panel["cgl_inventory_change"], panel["crush_surprise"], panel["crush_utilization"]],
}

generic_rows = []
for signal_set, families in {"A": families_A, "B": families_B}.items():
    family_features = pd.DataFrame({name: pd.concat(members, axis=1).mean(axis=1) for name, members in families.items()}, index=trading_index).fillna(0.0)
    target = futures_pnl[COMMODITY].shift(-1)
    tests = {
        "avg_all_signals": family_average(families, trading_index),
        "equal_family": equal_family(families, trading_index),
        "best_family_by_trend_mr": best_family_by_trend_mr(families, panel, trading_index),
        "select_by_ic": select_by_ic_signal(families, target, trading_index),
        "expanding_ols_family_model": prediction_to_signal(expanding_ols_prediction(family_features, target), trading_index),
        "kalman_family_model": prediction_to_signal(kalman_prediction(family_features, target), trading_index),
    }
    for name, signal in tests.items():
        bt = backtest_signal(signal, futures_pnl, COMMODITY)
        row = {"signal_set": signal_set}
        row.update(metric_row(name, bt))
        generic_rows.append(row)

generic_results = pd.DataFrame(generic_rows).sort_values(["signal_set", "validation_sharpe", "oos_sharpe"], ascending=[True, False, False])
display(generic_results.round(3))

,signal_set,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,turnover
2,A,best_family_by_trend_mr,long_short,0.180,0.758,0.758,"1,305.316",-11.056,0.480,0.006
3,A,select_by_ic,long_short,0.191,0.718,0.342,365.363,-7.606,0.378,0.004
5,A,kalman_family_model,long_short,-0.027,0.683,0.294,500.413,-14.819,0.250,0.006
4,A,expanding_ols_family_model,long_short,-0.693,0.637,0.282,442.340,-13.131,-0.012,0.006
1,A,equal_family,long_short,0.058,0.577,0.546,416.848,-7.438,0.306,0.002
0,A,avg_all_signals,long_short,0.078,0.486,0.635,511.507,-8.331,0.333,0.002
10,B,expanding_ols_family_model,long_short,-0.782,1.142,0.486,572.316,-5.800,0.153,0.006
8,B,best_family_by_trend_mr,long_short,0.180,0.758,0.758,"1,305.316",-11.056,0.480,0.006
11,B,kalman_family_model,long_short,-0.064,0.742,0.351,499.323,-8.386,0.267,0.006
6,B,avg_all_signals,long_short,-0.062,0.015,0.393,205.565,-4.164,0.081,0.002


## 4. Momentum/MR Price Benchmark

In [5]:
trend_gate = panel["mom_60"].abs() > panel["mom_60"].abs().expanding(min_periods=252).median().shift(1)
price_signals = {
    "mom20": clean_signal(panel["mom_20"], trading_index),
    "mom60": clean_signal(panel["mom_60"], trading_index),
    "mr5": clean_signal(panel["rev_5"], trading_index),
    "mom60_mr5_equal": clean_signal((panel["mom_60"] + panel["rev_5"]) / 2.0, trading_index),
    "trend_mom_else_mr": clean_signal(panel["mom_60"].where(trend_gate, panel["rev_5"]), trading_index),
}
price_rows = [metric_row(name, backtest_signal(signal, futures_pnl, COMMODITY)) for name, signal in price_signals.items()]
display(pd.DataFrame(price_rows).sort_values("oos_sharpe", ascending=False).round(3))

,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,turnover
1,mom60,long_short,0.465,0.666,0.981,"1,928.351",-12.034,0.665,0.003
4,trend_mom_else_mr,long_short,0.223,0.799,0.914,"1,849.798",-12.689,0.561,0.006
3,mom60_mr5_equal,long_short,0.211,0.401,0.807,859.294,-6.935,0.422,0.004
0,mom20,long_short,0.179,0.440,0.389,633.967,-10.798,0.293,0.005
2,mr5,long_short,-0.341,-0.302,-0.527,-616.086,-10.208,-0.386,0.008


## 5. Cargill Overlay Setup

Cargill is prepared before walk-forward. The promoted soybean path still treats Cargill as a physical disagreement haircut, not as a fitted model.

In [6]:
cargill_physical = clean_signal((-panel["cgl_inventory_change"] + panel["crush_surprise"] + panel["crush_utilization"]) / 3.0, trading_index)
pnl_vol = futures_pnl[COMMODITY].rolling(60, min_periods=20).std().shift(1)
low_vol = pnl_vol < (0.80 * pnl_vol.expanding(min_periods=252).median().shift(1))

cargill_adjusted_signals = {}
for name, signal in price_signals.items():
    disagreement = np.sign(signal) * np.sign(cargill_physical) < 0.0
    adjusted = clean_signal(signal.where(~disagreement, 0.50 * signal), trading_index)
    adjusted = clean_signal(adjusted.where(~low_vol.reindex(trading_index).fillna(False), 0.0), trading_index)
    cargill_adjusted_signals[name] = adjusted

cargill_preview = pd.DataFrame([
    metric_row(f"{name} + cargill_disagree_half + low_vol_flat", backtest_signal(signal, futures_pnl, COMMODITY))
    for name, signal in cargill_adjusted_signals.items()
]).sort_values("oos_sharpe", ascending=False)
display(cargill_preview.round(3))

,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,turnover
4,trend_mom_else_mr + cargill_disagree_half + lo...,long_short,0.417,0.014,3.074,"1,346.844",-3.503,0.746,0.004
1,mom60 + cargill_disagree_half + low_vol_flat,long_short,0.562,0.044,2.864,"1,235.495",-3.503,0.820,0.003
3,mom60_mr5_equal + cargill_disagree_half + low_...,long_short,0.433,0.509,2.461,603.443,-2.392,0.714,0.003
0,mom20 + cargill_disagree_half + low_vol_flat,long_short,0.328,0.031,1.248,336.110,-3.652,0.397,0.003
2,mr5 + cargill_disagree_half + low_vol_flat,long_short,-0.069,1.600,-1.242,-270.486,-4.561,-0.104,0.005


## 6. Annual Walk-Forward Momentum/MR

The walk-forward selector is run after the Cargill overlay setup, using the Cargill-aware Momentum/MR candidate menu.

In [7]:
candidate_backtests = {name: backtest_signal(signal, futures_pnl, COMMODITY) for name, signal in cargill_adjusted_signals.items()}
walk_forward_signal = pd.Series(0.0, index=trading_index)
wf_rows = []
for year in range(OOS_START.year, trading_index.max().year + 1):
    start = pd.Timestamp(f"{year}-01-01")
    end = pd.Timestamp(f"{year + 1}-01-01")
    validation_start = start - pd.DateOffset(years=2)
    train_mask = trading_index < validation_start
    validation_mask = (trading_index >= validation_start) & (trading_index < start)
    trade_mask = (trading_index >= start) & (trading_index < end)
    scored = []
    for name, bt in candidate_backtests.items():
        train = active_metrics(bt, train_mask)
        validation = active_metrics(bt, validation_mask)
        scored.append({"year": year, "candidate": name, "train_sharpe": train["sharpe"], "validation_sharpe": validation["sharpe"], "score": validation["sharpe"] if train["sharpe"] > 0 else -np.inf})
    score_table = pd.DataFrame(scored)
    selected = score_table.sort_values(["score", "validation_sharpe"], ascending=False).iloc[0]["candidate"]
    walk_forward_signal.loc[trade_mask] = cargill_adjusted_signals[selected].loc[trade_mask]
    trade = active_metrics(candidate_backtests[selected], trade_mask)
    wf_rows.append({"year": year, "selected": selected, "trade_sharpe": trade["sharpe"], "trade_pnl": trade["total_pnl"], "trade_dd_pct": trade["max_drawdown"] / DD_CAPITAL_USD * 100.0, "active_days": trade["days"]})

final_bt = backtest_signal(walk_forward_signal, futures_pnl, COMMODITY)
final_row = pd.DataFrame([metric_row("annual_walk_forward_momentum_mr + cargill_disagree_half + low_vol_flat", final_bt)])

display(pd.DataFrame(wf_rows).round(3))
display(final_row.round(3))
row = final_row.iloc[0]
display(Markdown(f"""
## Conclusion

Final soybean candidate: `annual_walk_forward_momentum_mr + cargill_disagree_half + low_vol_flat`.

OOS Sharpe `{row['oos_sharpe']:.3f}`, OOS PnL `{row['oos_pnl']:.3f}`, OOS max DD `{row['oos_dd_pct']:.3f}%`.

The generic A/B tests are kept as diagnostics. The promoted rule is the walk-forward Momentum/MR spine after the Cargill physical disagreement haircut is defined.
"""))

,year,selected,trade_sharpe,trade_pnl,trade_dd_pct,active_days
0,2018,mom60_mr5_equal,1.590,193.766,-2.392,107.000
1,2019,trend_mom_else_mr,-2.552,-109.248,-1.288,33.000
2,2020,trend_mom_else_mr,5.457,829.741,-1.671,57.000


,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,turnover
0,annual_walk_forward_momentum_mr + cargill_disa...,long_short,NaN,NaN,2.601,914.259,-2.392,2.601,0.004



## Conclusion

Final soybean candidate: `annual_walk_forward_momentum_mr + cargill_disagree_half + low_vol_flat`.

OOS Sharpe `2.601`, OOS PnL `914.259`, OOS max DD `-2.392%`.

The generic A/B tests are kept as diagnostics. The promoted rule is the walk-forward Momentum/MR spine after the Cargill physical disagreement haircut is defined.
